# 🚗 Did fuel-tax cuts get Europe driving again?
### A 3-hour taste of data-driven research

In early 2026 a war spiked oil prices across Europe. Sixteen governments rushed
out **fuel-tax cuts**. Did the cuts work? Did people change how they drove?

This is a real research question, and today you'll answer it with **real data** —
the same data behind a scientific paper written in this department. By the end
you will have:

1. **Collected** a piece of data yourself, live from the internet.
2. **Plotted** how an entire continent's driving changed over six weeks.
3. **Checked** whether our way of measuring "driving" is even trustworthy.
4. **Caught a trap** that fools journalists, politicians — and scientists.

---
**How this notebook works**
- Run a cell with **Shift + Enter**.
- Most cells are already written — just run them.
- A few cells are yours to finish. They are marked **`# TODO`** with a hint.
  Replace the `...` with your code, then run it.
- Stuck? Raise your hand. There are answers, but try first — that's the fun part.

In [ ]:
# ▶️ Run me first. Loads our tools and the workshop data.
#
# ┌─ INSTRUCTOR: for one-click Colab, paste a PUBLIC link to data.zip below. ──┐
# │ Works with a GitHub "raw" link OR a Google Drive share link.              │
# │ Leave it "" to use a local data/ folder, or to upload data.zip by hand.   │
DATA_URL = "https://raw.githubusercontent.com/Societal-Computing/fuel_impact_workshop/main/data.zip"
# └───────────────────────────────────────────────────────────────────────────┘

import io, json, re, pathlib, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

def _find_data():
    for p in [pathlib.Path("data"), pathlib.Path("workshop/data"), pathlib.Path(".")]:
        if (p / "busyness_daily.csv").exists():
            return p
    return None

def _download(url):
    if "drive.google.com" in url:                 # turn a Drive share link into a download
        m = re.search(r"/d/([\w-]+)", url) or re.search(r"[?&]id=([\w-]+)", url)
        file_id = m.group(1)
        try:
            import gdown                           # pre-installed on Colab
            gdown.download(id=file_id, output="data.zip", quiet=True)
            return pathlib.Path("data.zip").read_bytes()
        except Exception:
            url = f"https://drive.google.com/uc?export=download&id={file_id}"
    return requests.get(url, timeout=60).content

DATA = _find_data()
if DATA is None and DATA_URL:                      # one-click path: auto-download the data
    print("⬇️  Downloading the workshop data…")
    zipfile.ZipFile(io.BytesIO(_download(DATA_URL))).extractall(".")
    DATA = _find_data()
if DATA is None:                                   # fallback: manual upload on Colab
    try:
        from google.colab import files
        print("📂 Upload 'data.zip' (or the files inside the data folder):")
        up = files.upload()
        for name, content in up.items():
            if name.lower().endswith(".zip"):
                zipfile.ZipFile(io.BytesIO(content)).extractall(".")
        DATA = _find_data()
    except ImportError:
        pass
if DATA is None:
    raise FileNotFoundError("No workshop data found. Set DATA_URL above, run from the "
                            "folder that contains 'data/', or upload data.zip on Colab.")

print("✅ Ready. Using data from:", DATA.resolve())

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# 1 · Where does data come from?

Data doesn't appear by magic — **someone has to collect it.** There are two big
ways, and we'll taste both.

### 1a · Asking a computer that's listening (an *API*)
Some organisations run servers that answer questions over the internet. You send
a web address, they send back data. Let's ask **Open-Meteo** (a free weather
service — no sign-up needed) what the weather is *right now* in Saarbrücken,
where this department sits.

In [ ]:
# A web address that asks for the current weather at Saarbrücken (49.23°N, 6.99°E).
url = ("https://api.open-meteo.com/v1/forecast"
       "?latitude=49.23&longitude=6.99"
       "&current=temperature_2m,precipitation,wind_speed_10m"
       "&timezone=Europe/Berlin")

try:
    weather = requests.get(url, timeout=15).json()   # ask, and read the answer
    print("🌍 Fresh data, straight from the internet!\n")
except Exception as e:
    print("(No internet — using a saved copy.)\n")
    weather = json.load(open(DATA / "weather_example.json"))

# The answer comes back as JSON: nested boxes of values. Here it is:
print(json.dumps(weather, indent=2)[:550])

That blob is **JSON** — just boxes inside boxes (Python calls them *dictionaries*
and *lists*). To get one value out, you open the boxes one at a time:
`weather["current"]["temperature_2m"]`.

In [ ]:
# TODO: pull the current temperature out of the `weather` data into `temp`.
# Hint: look at the printout above. There's a "current" box, and inside it a
#       "temperature_2m" box. Open them in order.
temp = ...        # <-- replace the ...

print(f"🌡️  It is currently {temp}°C in Saarbrücken.")

### 1b · Watching the world (a *scraper*)
Weather is easy — someone built a tidy API for it. But **how busy is a gas
station right now?** Nobody publishes that as a neat feed. It *is* visible, though,
on Google Maps ("Live: busier than usual"). So the researchers wrote a program — a
**scraper** — that automatically checked thousands of places, **every hour, for
seven months**, and saved what it saw.

Here is **one real saved response**: what the scraper recorded for a single gas
station at a single moment.

In [ ]:
poi = json.load(open(DATA / "popular_times_example.json"))

print("Name:            ", poi["title"])
print("What kind:       ", poi["placeTypes"])
print("Captured at:     ", poi["scrapedAtLocal"], poi["timezone"])
print("Busy RIGHT NOW:  ", poi["popularTimesLivePercent"], "%   <-- the key number")

Google also gives a **"typical week"** — how busy this place *usually* is each
hour. That's the baseline the researchers compare the live number against. Let's
draw the typical **Friday**, with the live reading on top.

In [ ]:
friday = pd.DataFrame(poi["popularTimesHistogram"]["Friday"])

plt.figure()
plt.bar(friday["hour"], friday["occupancyPercent"], color="#cccccc", label="Typical Friday")
plt.axhline(poi["popularTimesLivePercent"], color="#d62728", lw=2.5,
            label=f"Live now: {poi['popularTimesLivePercent']:.0f}%")
plt.xlabel("Hour of day"); plt.ylabel("How busy (%)")
plt.title(poi["title"]); plt.legend(); plt.show()

**Pause and appreciate the scale.** That was *one* place at *one* moment. The full
dataset is thousands of places × every hour × seven months ≈ **25 million rows.**
No human could collect that by hand — which is exactly why we automate it. From
here on, we work with the cleaned-up table that all those scrapes became.

# 2 · Plot the V-shape

The 25 million scrapes have already been tidied into a simple table: **one row =
one country, one day, one type of place, and how busy it was on average.** Let's
load it and ask our real question: *did the oil shock change how much people drove?*

In [ ]:
busy = pd.read_csv(DATA / "busyness_daily.csv", parse_dates=["date"])
price = pd.read_csv(DATA / "diesel_price_daily.csv", parse_dates=["date"])

print(busy.shape[0], "rows.  Each row is one country, one day, one place-type.")
busy.head()

In [ ]:
# We start with the 15 countries that DID cut fuel taxes (we'll use the others later).
interveners = busy[busy["intervened"]].copy()
print("Countries that cut taxes:", interveners["country"].nunique())

Now your turn — the payoff cell. We want **one line for gas stations** and **one
for supermarkets**, averaged across all 15 countries, with the **diesel price**
drawn behind them. The plotting is written for you; you just have to build the
daily averages.

In [ ]:
# GOAL: build `daily`, a table with one row per date and two columns: "gas" and "supermarket".
#
# Step 1 — average busyness across countries for each (date, poi_class), then split
#          the two place-types into columns with .unstack("poi_class"):
#   daily = interveners.groupby(["date", "poi_class"])["live_pct"].mean().unstack("poi_class")
#
# Step 2 — smooth the weekday wiggle with a 7-day rolling average:
#   daily = daily.rolling(7, center=True, min_periods=4).mean()

# TODO: write Step 1 and Step 2 (replace the ...)
daily = ...


# ---- everything below is done for you; it runs once `daily` exists ----
fig, ax = plt.subplots()
ax.plot(daily.index, daily["gas"], lw=2.2, color="#1f77b4", label="Gas stations")
ax.plot(daily.index, daily["supermarket"], lw=2.2, color="#2ca02c", label="Supermarkets")
ax.set_ylabel("Live busyness (%)"); ax.legend(loc="lower right")
ax2 = ax.twinx()
ax2.plot(price["date"], price["diesel_price_eur_per_1000l"], "r--", alpha=.6)
ax2.set_ylabel("Diesel price (€ / 1000 L)", color="r")
plt.title("Did the oil shock change how much people drove?")
plt.show()

🎉 **You just reproduced a figure from a real scientific paper.** Read the shape:
as the **diesel price** (red) climbs to its peak around **1 April**, busyness
**dips**; as the price eases, busyness **rebounds**. A clear **V**.

And notice the green line: **supermarkets did the same thing as gas stations.**
So this isn't only about refuelling — people re-timed *all kinds* of car trips.
Keep that in mind; it becomes important soon.

# 3 · Is our measurement even real?

Here's the habit that separates a scientist from someone who just makes charts.
You found a beautiful pattern. **But should you trust it?** "Google busyness" is a
guess Google makes about crowds. What if it's junk, and the V-shape is an artefact?

> *"I found a pattern — but is my measurement even real?"*

The only way to know is to compare it against a **completely independent**
measurement of the same thing. Germany has physical sensors buried in the
motorways (*Autobahn* loop detectors) that literally **count cars**. If our
Google busyness rises and falls on the same days as the real car counts, we can
believe it. Let's load both.

In [ ]:
val = pd.read_csv(DATA / "validation_daily.csv", parse_dates=["date"])
# busyness = our Google signal: avg busyness at German gas stations that day
# flow     = the truth: avg cars-per-hour counted by motorway sensors that day
# speed    = avg car speed on the motorway
# dow      = day of week (0 = Monday ... 6 = Sunday)
val.head()

In [ ]:
# Plumbing — just run this. Weekends are busy everywhere, which could fake an
# agreement between the two signals. We strip out that weekly rhythm from BOTH,
# leaving only the genuine day-to-day ups and downs.
def remove_weekly_rhythm(series, dow):
    return series - dow.map(series.groupby(dow).mean())

val["busyness_clean"] = remove_weekly_rhythm(val["busyness"], val["dow"])
val["flow_clean"]     = remove_weekly_rhythm(val["flow"],     val["dow"])
print("Done — added 'busyness_clean' and 'flow_clean'.")

Now measure the agreement. **Correlation** is one number from −1 to +1: near +1
means "they rise and fall together", near 0 means "no relationship". Your job is
to compute it and *see* it.

In [ ]:
# TODO 1: compute the correlation between the two cleaned signals.
#   Hint: a pandas column has a .corr() method ->  val["A"].corr(val["B"])
r = ...

print(f"Correlation between Google busyness and real car counts:  r = {r:.2f}")

# TODO 2: draw a scatter plot — flow_clean on the x-axis, busyness_clean on the y-axis.
#   Hint: plt.scatter(x_values, y_values, alpha=0.7)
plt.figure()
...        # <-- your scatter plot here
plt.xlabel("Real motorway car counts (weekly rhythm removed)")
plt.ylabel("Google gas-station busyness (weekly rhythm removed)")
plt.title(f"Do two independent measurements agree?   r = {r:.2f}")
plt.show()

You should get **r ≈ 0.45**. Think about what that means: a number Google *guesses*
about crowds, and physical sensors that *count cars on a different part of the
country* — two totally separate systems — **move up and down together.** Our
busyness signal is measuring something real.

That's the most important move in all of data science: **before you trust a
number, check it against something independent.** A pattern you can't validate is
just a pretty picture.

# 4 · The trap: did the tax cuts cause it?

Now the dangerous part. Sixteen governments cut fuel taxes during exactly these
weeks. The obvious story writes itself: *taxes cut → fuel cheaper → people drive
more → the rebound.* Let's check the "naive" version a journalist might run.

In [ ]:
gas = busy[busy["poi_class"] == "gas"]
before = gas[gas["date"] < "2026-03-20"].groupby("intervened")["live_pct"].mean()
after  = gas[gas["date"] > "2026-05-01"].groupby("intervened")["live_pct"].mean()

print(f"Tax-cutting countries were  {before[True]:.0f}% busy BEFORE the cuts")
print(f"                    ... and {after[True]:.0f}% busy AFTER  the cuts")
print(f"\n👉 A jump of +{after[True]-before[True]:.0f} points. 'The cuts worked!'")

A reporter could now write: *"After Europe slashed fuel taxes, station traffic
surged. The policy worked."* The actual researchers ran a careful statistical
model and got the same flavour of answer: **+6 to +7 points, highly significant.**
Case closed?

**Not so fast.** Let's look at three countries that cut **no tax at all** —
France, the United Kingdom, and the Netherlands. If the *cuts* caused the rebound,
these three should look **different**.

In [ ]:
noncut = ["France", "United Kingdom", "Netherlands"]

# TODO: for each country in `noncut`, plot its daily GAS-station busyness over time
#       (7-day smoothed, so it's readable).
#   Hint: loop over noncut. For each country, take the rows where
#         busy["country"] == country AND busy["poi_class"] == "gas",
#         sort by date, smooth with .rolling(7, center=True, min_periods=4).mean(),
#         and plt.plot(dates, values, label=country).
plt.figure()
for country in noncut:
    ...     # filter -> smooth -> plot this country's gas busyness

plt.ylabel("Gas-station busyness (%)")
plt.title("Three countries that cut NO fuel tax")
plt.legend(); plt.show()

😮 **The exact same V-shape.** France, the UK, and the Netherlands cut nothing, yet
their driving dipped and rebounded *just like* the cutting countries. So the
rebound was **not caused by the tax cuts** — it was the shared shock (and the
Easter holidays) happening to *everyone* at once.

"After the cuts, things got better" is **not** the same as "the cuts *made* things
better." The countries that did nothing are the **control group**, and they prove
it. This single mistake — confusing *what happened at the same time* with *what
caused it* — is one of the most common and expensive errors in the world.

---
### 🗣️ Your debate (split into two pairs)
- **Pair A:** Argue that the tax cuts worked. What evidence supports you? What
  would you need to *prove* it?
- **Pair B:** Argue that you *cannot* claim the cuts worked. Use the three
  no-cut countries. What's the strongest version of A's case you still have to beat?

There's no trick answer. The real paper landed on Pair B's side — *honestly
reporting that it could not credit the cuts* — and that honesty became its main
scientific contribution.

# 🏁 What you just did

In three hours you ran a real research project:

1. **Collected** data — a live API call and a real scraped record.
2. **Found a pattern** — the continent-wide V-shape in driving.
3. **Validated it** — checked your signal against independent car-counting sensors (r ≈ 0.45).
4. **Resisted the easy story** — used a control group to avoid claiming the cuts caused the rebound.

That loop — *collect → visualise → **validate** → question causation* — is how you
answer a societal question with data instead of with opinions. The data doesn't
hand you the answer; **how carefully you question it** is the whole job.